# Matching and retrieval experiments

Archived from the former Python scripts/package so the Hermes experiments are kept as notebooks.


## Exact and Fuzzy Matching

Lookup and string-similarity methods for repeated or near-repeated literals.

_Archived from `src/icd10_coding/retrieval/matching.py`._


In [ ]:
"""Step 1: exact and fuzzy literal matching."""

import time

import pandas as pd
from rapidfuzz import fuzz, process
from sklearn.model_selection import train_test_split

from icd10_coding.paths import DATA_PROCESSED_DIR, PREDICTIONS_DIR, ensure_output_dirs


FUZZY_THRESHOLD = 60


def build_exact_lookup(train: pd.DataFrame) -> dict[str, str]:
    exact_lookup = {}
    for lit_norm, group in train.groupby("Literal_norm"):
        exact_lookup[lit_norm] = group["Code"].value_counts().index[0]
    return exact_lookup


def run() -> None:
    ensure_output_dirs()

    train = pd.read_csv(DATA_PROCESSED_DIR / "train_preprocessed.csv")
    test = pd.read_csv(DATA_PROCESSED_DIR / "test_preprocessed.csv")

    exact_lookup = build_exact_lookup(train)
    test["exact_code"] = test["Literal_norm"].map(exact_lookup)

    print(f"Unique normalized literals in training: {len(exact_lookup):,}")
    print(f"Exact matches: {test['exact_code'].notna().sum():,}")

    train_literals = list(exact_lookup.keys())
    unmatched = test[test["exact_code"].isna()].copy()
    fuzzy_results = []

    print(f"Running fuzzy matching on {len(unmatched):,} unmatched test literals")
    start = time.time()
    for idx, row in unmatched.iterrows():
        result = process.extractOne(
            row["Literal_norm"],
            train_literals,
            scorer=fuzz.ratio,
            score_cutoff=0,
        )
        if result is not None:
            best_literal, score, _ = result
            fuzzy_results.append(
                {
                    "test_idx": idx,
                    "test_literal": row["Literal_norm"],
                    "matched_literal": best_literal,
                    "matched_code": exact_lookup[best_literal],
                    "fuzzy_score": score,
                }
            )

    fuzzy_df = pd.DataFrame(fuzzy_results)
    print(f"Done in {time.time() - start:.1f} seconds")

    train_split, val_split = train_test_split(train, test_size=0.2, random_state=42)
    val_lookup = build_exact_lookup(train_split)
    val_split = val_split.copy()
    val_split["exact_pred"] = val_split["Literal_norm"].map(val_lookup)
    val_matched = val_split[val_split["exact_pred"].notna()]
    val_exact_correct = (val_matched["exact_pred"] == val_matched["Code"]).sum()
    print(
        f"Exact validation matches: {len(val_matched):,} / {len(val_split):,}; "
        f"correct {val_exact_correct:,}"
    )

    val_unmatched = val_split[val_split["exact_pred"].isna()]
    val_train_literals = list(val_lookup.keys())
    val_fuzzy = []
    for idx, row in val_unmatched.iterrows():
        result = process.extractOne(
            row["Literal_norm"],
            val_train_literals,
            scorer=fuzz.ratio,
            score_cutoff=0,
        )
        if result is not None:
            val_fuzzy.append(
                {
                    "val_idx": idx,
                    "matched_code": val_lookup[result[0]],
                    "score": result[1],
                    "true_code": val_split.loc[idx, "Code"],
                }
            )
    val_fuzzy_df = pd.DataFrame(val_fuzzy)
    results_by_threshold = []
    for threshold in [50, 55, 60, 65, 70, 75, 80, 85, 90, 95]:
        accepted = val_fuzzy_df[val_fuzzy_df.score >= threshold]
        n_correct = (accepted["matched_code"] == accepted["true_code"]).sum()
        overall_acc = (val_exact_correct + n_correct) / len(val_split) * 100
        results_by_threshold.append((threshold, len(accepted), n_correct, overall_acc))
    best = max(results_by_threshold, key=lambda x: x[3])
    print(f"Best validation threshold: {best[0]} (overall accuracy {best[3]:.1f}%)")

    test["step1_code"] = test["exact_code"]
    test["step1_method"] = test["exact_code"].apply(
        lambda x: "exact" if pd.notna(x) else "unresolved"
    )
    for _, fz in fuzzy_df.iterrows():
        if fz["fuzzy_score"] >= FUZZY_THRESHOLD:
            test.loc[fz["test_idx"], "step1_code"] = fz["matched_code"]
            test.loc[fz["test_idx"], "step1_method"] = "fuzzy"

    output = test[
        ["id", "Literal", "Literal_clean", "Literal_norm", "step1_code", "step1_method"]
    ].copy()
    output.to_csv(PREDICTIONS_DIR / "step1_predictions.csv", index=False)

    print("Step 1 predictions:")
    print(test["step1_method"].value_counts())
    print("Saved step1_predictions.csv")


if __name__ == "__main__":
    run()


## TF-IDF Retrieval

Literal and reference retrieval experiments based on sparse text similarity.

_Archived from `src/icd10_coding/retrieval/tfidf_retrieval.py`._


In [ ]:
"""Step 2: TF-IDF retrieval over ICD descriptions."""

import time

import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

from icd10_coding.paths import DATA_PROCESSED_DIR, PREDICTIONS_DIR, ensure_output_dirs
from icd10_coding.retrieval.matching import build_exact_lookup


BATCH = 100


def build_retrieval_index(reference: pd.DataFrame):
    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
    )
    word_vectorizer = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
    )
    char_ref_vectors = char_vectorizer.fit_transform(reference["Description_norm"])
    word_ref_vectors = word_vectorizer.fit_transform(reference["Description_norm"])
    return char_vectorizer, word_vectorizer, char_ref_vectors, word_ref_vectors


def retrieve_codes(texts, reference, char_vectorizer, word_vectorizer, char_ref_vectors, word_ref_vectors):
    test_char_vectors = char_vectorizer.transform(texts)
    test_word_vectors = word_vectorizer.transform(texts)

    codes, descriptions, combined_scores, char_scores, word_scores = [], [], [], [], []
    for i in range(0, len(texts), BATCH):
        end = min(i + BATCH, len(texts))
        char_sims = cosine_similarity(test_char_vectors[i:end], char_ref_vectors)
        word_sims = cosine_similarity(test_word_vectors[i:end], word_ref_vectors)
        combined_sims = (char_sims + word_sims) / 2
        best_idx = combined_sims.argmax(axis=1)
        for j in range(end - i):
            bi = best_idx[j]
            codes.append(reference.iloc[bi]["Code"])
            descriptions.append(reference.iloc[bi]["Description"])
            combined_scores.append(float(combined_sims[j, bi]))
            char_scores.append(float(char_sims[j, bi]))
            word_scores.append(float(word_sims[j, bi]))
        del char_sims, word_sims, combined_sims
    return codes, descriptions, combined_scores, char_scores, word_scores


def run() -> None:
    ensure_output_dirs()

    train = pd.read_csv(DATA_PROCESSED_DIR / "train_preprocessed.csv")
    test = pd.read_csv(DATA_PROCESSED_DIR / "test_preprocessed.csv")
    reference = pd.read_csv(DATA_PROCESSED_DIR / "reference_preprocessed.csv")
    step1 = pd.read_csv(PREDICTIONS_DIR / "step1_predictions.csv")

    reference["Description_norm"] = reference["Description_norm"].fillna("")
    test["Literal_norm"] = test["Literal_norm"].fillna("")

    print("Building TF-IDF retrieval indexes...")
    start = time.time()
    char_vec, word_vec, char_ref, word_ref = build_retrieval_index(reference)
    print(f"Built in {time.time() - start:.1f}s")

    print(f"Running retrieval on {len(test):,} test literals")
    start = time.time()
    ret = retrieve_codes(test["Literal_norm"], reference, char_vec, word_vec, char_ref, word_ref)
    (
        test["retrieval_code"],
        test["retrieval_desc"],
        test["retrieval_score"],
        test["retrieval_char_score"],
        test["retrieval_word_score"],
    ) = ret
    print(f"Done in {time.time() - start:.1f}s")

    train["Literal_norm"] = train["Literal_norm"].fillna("")
    train_split, val_split = train_test_split(train, test_size=0.2, random_state=42)
    val_split = val_split.copy()
    val_ret = retrieve_codes(
        val_split["Literal_norm"], reference, char_vec, word_vec, char_ref, word_ref
    )
    val_split["ret_code"] = val_ret[0]
    val_split["ret_score"] = val_ret[2]
    val_split["ret_correct"] = val_split["ret_code"] == val_split["Code"]

    step1_lookup = build_exact_lookup(train_split)
    val_split["s1_exact"] = val_split["Literal_norm"].map(step1_lookup)
    val_unmatched = val_split[val_split["s1_exact"].isna()]
    lookup_keys = list(step1_lookup.keys())
    fuzzy_codes = {}
    for idx, row in val_unmatched.iterrows():
        result = process.extractOne(row["Literal_norm"], lookup_keys, scorer=fuzz.ratio, score_cutoff=60)
        if result is not None:
            fuzzy_codes[idx] = step1_lookup[result[0]]
    val_split["s1_code"] = val_split["s1_exact"].copy()
    for idx, code in fuzzy_codes.items():
        val_split.loc[idx, "s1_code"] = code
    val_split["s1_correct"] = val_split["s1_code"] == val_split["Code"]
    val_split["combined_code"] = val_split["s1_code"]
    missing = val_split["combined_code"].isna()
    val_split.loc[missing, "combined_code"] = val_split.loc[missing, "ret_code"]
    val_split["combined_correct"] = val_split["combined_code"] == val_split["Code"]

    print("Validation comparison")
    print(f"Step 1 only: {val_split.s1_correct.mean() * 100:.1f}%")
    print(f"Step 2 only: {val_split.ret_correct.mean() * 100:.1f}%")
    print(f"Combined:    {val_split.combined_correct.mean() * 100:.1f}%")

    test = test.merge(step1[["id", "step1_code", "step1_method"]], on="id", how="left")
    test["final_code"] = test["step1_code"]
    test["final_method"] = test["step1_method"]
    needs_fallback = test["final_code"].isna()
    test.loc[needs_fallback, "final_code"] = test.loc[needs_fallback, "retrieval_code"]
    test.loc[needs_fallback, "final_method"] = "retrieval"

    output_cols = [
        "id", "Literal", "Literal_clean", "Literal_norm",
        "step1_code", "step1_method",
        "retrieval_code", "retrieval_desc",
        "retrieval_score", "retrieval_char_score", "retrieval_word_score",
        "final_code", "final_method",
    ]
    test[output_cols].to_csv(PREDICTIONS_DIR / "step2_predictions.csv", index=False)
    print("Saved step2_predictions.csv")


if __name__ == "__main__":
    run()


## Matching Experiment Runner

Former command-line wrapper for the matching experiments.

_Archived from `scripts/step1_matching.py`._


In [ ]:
from _bootstrap import PROJECT_ROOT  # noqa: F401
from icd10_coding.retrieval.matching import run


if __name__ == "__main__":
    run()


## Retrieval Experiment Runner

Former command-line wrapper for retrieval experiments.

_Archived from `scripts/step2_retrieval.py`._


In [ ]:
from _bootstrap import PROJECT_ROOT  # noqa: F401
from icd10_coding.retrieval.tfidf_retrieval import run


if __name__ == "__main__":
    run()
